# Embeddings & Vector Store
---

# Información

## Objetivo

Generar embeddings y construir un vector store utilizando información oficial del Plan Comunal de Emergencia de Valdivia.

## Objetivos específicos

- Cargar el documento limpio generado en la etapa de ingestión.
- Crear fragmentos de texto para búsqueda semántica.
- Transformar los fragmentos en embeddings.
- Construir un índice vectorial.

## Entradas

- data/processed/Plan-Comunal-de-Emergencia-2025-2.json

## Salidas

- Embeddings generados.
- Vector store para recuperación semántica.



## Etapa 1: Generación de Embeddings

En esta etapa se transformará el contenido textual de cada `Chunk` en una representación vectorial utilizando un modelo de Sentence Transformers.

Los embeddings permiten representar el significado semántico del texto mediante vectores numéricos. Estas representaciones constituyen la base de la búsqueda semántica utilizada por el sistema RAG.

### 1. Importación de dependencias.

In [76]:
from pathlib import Path
import sys

# Agregar la raíz del proyecto al PATH para importar módulos de src
project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [78]:
from sentence_transformers import SentenceTransformer

from src.ingestion.loaders import load_pdf
from src.processing.cleaner import clean_document
from src.processing.embedder import generate_embeddings
import src.processing.chunker


### 2. Carga del documento

En esta sección se carga un documento PDF de prueba utilizando el módulo `load_pdf`, desarrollado durante la etapa de Ingesta. El objetivo es obtener un objeto `Document` que servirá como punto de partida para el resto del pipeline.

Para esta validación se utilizará el documento **test_plan_comunal.pdf**, el mismo empleado en las pruebas de las etapas anteriores, garantizando la consistencia durante el desarrollo del proyecto.

In [79]:
import json
from pathlib import Path


processed_file = Path(
    "../data/processed/Plan-Comunal-de-Emergencia-2025-2.json"
)

with processed_file.open(
    "r",
    encoding="utf-8"
) as file:
    document_data = json.load(file)


document_data.keys()

dict_keys(['id', 'title', 'source', 'file_type', 'content', 'metadata'])

In [80]:
from src.models.document import Document


document = Document(
    id=document_data["id"],
    title=document_data["title"],
    source=document_data["source"],
    file_type=document_data["file_type"],
    content=document_data["content"],
    metadata=document_data["metadata"]
)

print(f"Título: {document.title}")
print(f"Fuente: {document.source}")
print(f"Tipo de archivo: {document.file_type}")
print(f"Longitud del contenido: {len(document.content)} caracteres")
print(f"Metadatos: {document.metadata}")



Título: Plan-Comunal-de-Emergencia-2025-2
Fuente: ..\data\raw\muni_valdivia\planes\Plan-Comunal-de-Emergencia-2025-2.pdf
Tipo de archivo: pdf
Longitud del contenido: 90189 caracteres
Metadatos: {'pages': 33}


---
## Etapa 2. Preparación de chunks

In [97]:
import src.processing.chunker as chunker

print(chunker.__file__)
print(hasattr(chunker, "create_chunks"))
print(dir(chunker))

c:\ALURA_AI_Proyect\Challege_Alura_ONE_AI_FOR_TECH\Challenge_Alura_Agente\src\processing\chunker.py
False
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__']


In [101]:
import src.processing.chunker as chunker

print(chunker.__file__)
print(dir(chunker))

c:\ALURA_AI_Proyect\Challege_Alura_ONE_AI_FOR_TECH\Challenge_Alura_Agente\src\processing\chunker.py
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__']
